In [1]:
import requests
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_ollama import OllamaEmbeddings

In [2]:
DOCS_BASE = "https://docs.langchain.com"

DOC_PATHS = [
    "oss/python/langchain/agents",
    "oss/python/deepagents/rag",
    "oss/python/langchain/tools",
    "oss/python/langchain/models",
    "oss/python/langchain/retrieval",
    "oss/python/langchain/knowledge-base",
    "oss/python/langchain/middleware",
    "oss/python/deepagents/overview",
    "oss/python/deepagents/subagents",
    "oss/python/deepagents/streaming",
    "oss/python/deepagents/frontend/subagent-streaming",
    "oss/python/deepagents/backends",
    "oss/python/langgraph/overview",
    "oss/python/langgraph/quickstart",
]

In [3]:
def load_langchain_docs(doc_paths: list[str] | None = None) -> list[Document]:
    paths = doc_paths or DOC_PATHS
    docs: list[Document]=[]
    for path in paths:
        url = f"{DOCS_BASE}/{path}.md"
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
        except requests.RequestException:
            continue
        source = f"{DOCS_BASE}/{path}"
        docs.append(
            Document(page_content=response.text, metadata={"source":source})
        )
    return docs

In [4]:
docs = load_langchain_docs()
print(f"Loaded {len(docs)} documentation pages.")

Loaded 14 documentation pages.


In [5]:
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"Total characters: {total_chars}")
print(docs[0].page_content[:500])

Total characters: 689058
> ## Documentation Index
> Fetch the complete documentation index at: https://docs.langchain.com/llms.txt
> Use this file to discover all available pages before exploring further.

# Agents

An agent is a model calling tools in a loop until a given task is complete.

<img src="https://mintcdn.com/langchain-5e9cc07a/jtty0O--UJOKG0nK/oss/images/core_agent_loop.svg?fit=max&auto=format&n=jtty0O--UJOKG0nK&q=85&s=4b4cbb497b6273758a565de1bc90ece0" alt="Core agent loop diagram" style={{height: "300px", 


In [13]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
print(f"Split documentation into {len(all_splits)} chunks.")

Split documentation into 931 chunks.


In [11]:
embeddings = OllamaEmbeddings(model='qwen3-embedding:0.6b')

In [12]:
client = QdrantClient(":memory:")

vector_size = len(embeddings.embed_query("sample text"))

if not client.collection_exists("test"):
    client.create_collection(
        collection_name="test",
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )
vector_store = QdrantVectorStore(
    client=client,
    collection_name="test",
    embedding=embeddings,
)

In [14]:
vector_store.add_documents(documents=all_splits)
print(f"Indexed {len(all_splits)} chunks.")

Indexed 931 chunks.


In [27]:
retrieved_docs = vector_store.similarity_search("what is the difference between agent.stream and agent.stream_events", k=20)

In [28]:
print(retrieved_docs)

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/subagents', '_id': '2c9a6e602c994117b5d3ca907eee4598', '_collection_name': 'test'}, page_content='## Streaming\n\nDeep Agents support streaming updates from both the coordinator and every delegated subagent.\n\nUse [`stream_events`](/oss/python/deepagents/event-streaming) to get typed projections—separate iterators for subagents, messages, tool calls, and values—so you can consume each independently.\n\n### Stream subagent progress\n\nThe simplest pattern is to iterate `stream.subagents` to track each delegated task as it starts, runs, and completes. Each subagent handle exposes `.name`, `.messages`, `.tool_calls`, and `.output`.\n\n<CodeGroup>\n  ```python Google theme={"theme":{"light":"catppuccin-latte","dark":"catppuccin-mocha"}}\n  from deepagents import (\n      create_deep_agent\n  )'), Document(metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/frontend/subagent-streaming', '_id':

In [31]:
count=0
for i in retrieved_docs:
    print(count)
    print(i.page_content[:500])
    count+=1

0
## Streaming

Deep Agents support streaming updates from both the coordinator and every delegated subagent.

Use [`stream_events`](/oss/python/deepagents/event-streaming) to get typed projections—separate iterators for subagents, messages, tool calls, and values—so you can consume each independently.

### Stream subagent progress

The simplest pattern is to iterate `stream.subagents` to track each delegated task as it starts, runs, and completes. Each subagent handle exposes `.name`, `.messages`
1
## Setting up `useStream`

No extra stream options are required. Point the stream at your deep agent,
render coordinator messages from `stream.messages`, and use `stream.subagents`
to mount cards for active specialists. In chat layouts, index subagents by the
tool-call ID that spawned them so each card appears under the coordinator turn
Point the stream at your deep agent, render coordinator messages from `stream.messages`, and use `stream.subagents` to mount cards for active specialists. I